In [10]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('analysis_avgx_2')

projs=['P0', 'Px', 'Py', 'Pz']
inserts=['tt', 'tx', 'ty', 'tz', 'xx', 'xy', 'xz', 'yy', 'yz', 'zz']
enss=['b','c','d','e']

tftcphy_A20_conn=tftcphy_B20_conn=(0.6,0.2)
tftcphy_A20_discq=tftcphy_A20_gluon=tftcphy_B20_discq=tftcphy_B20_gluon=(0.7,0.3)

mpl.rcParams['lines.markersize'] = mpl.rcParams['errorbar.capsize'] = 13
mpl.rcParams['axes.labelsize'] = mpl.rcParams['axes.titlesize'] = mpl.rcParams['xtick.labelsize'] = mpl.rcParams['ytick.labelsize'] = 40
mpl.rcParams['font.size'] = 20
yu.mpl_global_elinewidth=yu.mpl_global_capthick=3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
[ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]=yu.load_pkl_reg('ens2pars_jk_meffnst_selected',pathlabel='analysis_c2pt')
ens2msq2pars_jk=yu.load_pkl_reg('ens2msq2pars_jk',pathlabel='analysis_c2pt')

In [12]:
ens2Njk={'b':725,'c':400,'d':493,'e':460}
path='data_aux/RCs.pkl'
with open(path,'rb') as f:
    ens2RCs_me=pickle.load(f)
ens2RCs={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs[ens][key]=yu.jackknife_pseudo(ens2RCs_me[ens][key],ens2RCs_me[ens][f'{key}_err']*0+1e-10,ens2Njk[ens])[:,0]
        
path='data_aux/RCs_np.pkl'
with open(path,'rb') as f:
    ens2RCs_np_me=pickle.load(f)
ens2RCs_np={ens:{} for ens in enss}
for ens in enss:
    for key in ens2RCs_np_me[ens]:
        if key.endswith('err'):
            continue
        ens2RCs_np[ens][key]=yu.jackknife_pseudo(ens2RCs_np_me[ens][key],ens2RCs_np_me[ens][f'{key}_err']+1e-10,ens2Njk[ens])[:,0]

# conn

In [13]:
key2tf2ratio={}
for ens in enss:
    key2tf2ratio[(ens,'j+;conn')]={}
    key2tf2ratio[(ens,'j-;conn')]={}
    
    basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
    
    mN_jk=ens2pars_jk_meff2st[ens][:,0]
    
    factor_equal=1/(-3*mN_jk/4)
    
    if ens in ['e']:
        tf2c2pt={}
        path=f'{basepath}/conn_2pt_cfgs_sup_conn1D_run1.h5'
        with h5py.File(path) as f:
            moms=yu.moms2list(f['moms'])
            imom=moms.index([0,0,0])
            for tf in f['data'].keys():
                t=f[f'data/{tf}'][:]
                t=yu.jackknife(np.real(t[:,:,imom]))
                t=yu.superjackknife(t,yum.key2cfgs[(ens,'conn1D_run1')],yum.key2cfgs[(ens,'all')])
                tf2c2pt[int(tf)]=t
        path=f'{basepath}/conn_2pt_cfgs_sup_conn1D_run2.h5'
        with h5py.File(path) as f:
            moms=yu.moms2list(f['moms'])
            imom=moms.index([0,0,0])
            for tf in f['data'].keys():
                t=f[f'data/{tf}'][:]
                t=yu.jackknife(np.real(t[:,:,imom]))
                t=yu.superjackknife(t,yum.key2cfgs[(ens,'conn1D_run2')],yum.key2cfgs[(ens,'all')])
                tf2c2pt[int(tf)]=t

        path=f'{basepath}/conn_0,0,0,0,0,0_cfgs_sup_conn1D_run1.h5'
        with h5py.File(path) as f:
            moms=yu.moms2list(f['moms'])
            imom=moms.index([0,0,0,0,0,0])
            
            for jtf in f['data'].keys():
                j,tf=jtf.split('_'); tf=int(tf)
                t=f[f'data/{jtf}'][:]
                t=t[:,:,0,projs.index('P0'),inserts.index('tt')]
                t=yu.jackknife(t)
                t=yu.superjackknife(t,yum.key2cfgs[(ens,'conn1D_run1')],yum.key2cfgs[(ens,'all')])
                c3pt=t
                ratio=np.real(c3pt/tf2c2pt[tf][:,tf:tf+1]*factor_equal[:,None])
                key=(ens,j)
                key2tf2ratio[key][tf]=ratio
        path=f'{basepath}/conn_0,0,0,0,0,0_cfgs_sup_conn1D_run2.h5'
        with h5py.File(path) as f:
            moms=yu.moms2list(f['moms'])
            imom=moms.index([0,0,0,0,0,0])
            
            for jtf in f['data'].keys():
                j,tf=jtf.split('_'); tf=int(tf)
                t=f[f'data/{jtf}'][:]
                t=t[:,:,0,projs.index('P0'),inserts.index('tt')]
                t=yu.jackknife(t)
                t=yu.superjackknife(t,yum.key2cfgs[(ens,'conn1D_run2')],yum.key2cfgs[(ens,'all')])
                c3pt=t
                ratio=np.real(c3pt/tf2c2pt[tf][:,tf:tf+1]*factor_equal[:,None])
                key=(ens,j)
                key2tf2ratio[key][tf]=ratio
    else:
        path=f'{basepath}/conn_2pt.h5'
        with h5py.File(path) as f:
            moms=yu.moms2list(f['moms'])
            imom=moms.index([0,0,0])
            
            tf2c2pt={}
            for tf in f['data'].keys():
                t=f[f'data/{tf}'][:]
                t=yu.jackknife(np.real(t[:,:,imom]))
                tf2c2pt[int(tf)]=t

        path=f'{basepath}/conn_0,0,0,0,0,0.h5'
        with h5py.File(path) as f:
            moms=yu.moms2list(f['moms'])
            imom=moms.index([0,0,0,0,0,0])
            
            for jtf in f['data'].keys():
                j,tf=jtf.split('_'); tf=int(tf)
                t=f[f'data/{jtf}'][:]
                t=t[:,:,0,projs.index('P0'),inserts.index('tt')]
                c3pt=yu.jackknife(t)
                ratio=np.real(c3pt/tf2c2pt[tf][:,tf:tf+1]*factor_equal[:,None])
                key=(ens,j)
                key2tf2ratio[key][tf]=ratio
            
ens2tfs_conn={}
for ens in enss:
    tfs=list(key2tf2ratio[(ens,'j+;conn')].keys()); tfs.sort()
    ens2tfs_conn[ens]=tfs
    print(ens,tfs)

b [8, 10, 12, 14, 16, 18, 20]
c [6, 8, 10, 12, 14, 16, 18, 20, 22]
d [8, 10, 12, 14, 16, 18, 20, 22, 24, 26]
e [8, 11, 14, 17, 20, 23, 26, 29]


In [14]:
# plot

key2bare={}

overwrite=False
symmetrizeQ=True
def createDic(key):
    ens,j=key
    gett=lambda t:round(t/yu.ens2a[ens])
    gett2=lambda t:round(t/yu.ens2a[ens]/2)*2
    
    tf2ratio=key2tf2ratio[(ens,j)]
    tfmins_2st=ens2tfs_conn[ens][:-2]; tcmins_2st=range(2,ens2tfs_conn[ens][-1]//2-1)
    pars_jk_meff2st=ens2pars_jk_meff2st[ens]
    # if ens=='e':
    #     pars_jk_meff2st=yu.load_pkl_reg('ens2pars_jk_meffnst_selected',pathlabel='analysis_conn_Eensemble_withLessCfg')[1][ens]
        
    fittype='2st2step_SYMshare'
    
    if symmetrizeQ:
        tf2ratio=yu.symmetrizeRatio(tf2ratio)
    fits_sum=yu.doFits_3pt('sum',tf2ratio,tfmins_2st,tcmins_2st,corrQ=False,label=f'{ens}_{j}_sum',overwrite=overwrite)
    fits_sum=[fit for fit in fits_sum if fit[0][1]==2]
    fits_2st=yu.doFits_3pt(fittype,tf2ratio,tfmins_2st,tcmins_2st,pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,label=f'{ens}_{j}_2st_{symmetrizeQ}',overwrite=overwrite)

    # print(tfmins_2st,tcmins_2st)
    # print(len(fits_2st))
    
    (tfphy,tcphy)=tftcphy_A20_conn
    fits=fits_2st
    fitlabels=[fit[0] for fit in fits]
    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    fits_2st=[fit for fit in fits_2st if fit[0][1]==fits_2st[ind][0][1]]
    fit_2st_MA=yu.doMA_3pt(fits_2st,fitlabels=fitlabels[ind])

    key2bare[(ens,j)]=fit_2st_MA[0][:,0]
    
    yunit=ens2RCs_np_me[ens]['Zqq(mu=nu)']/ens2RCs_np_me[ens]['Zqq(mu!=nu)'] if j in ['j-;conn'] else 1
    # print(j,yunit)
    dic={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,None,None,fits_sum,fits_2st],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,fit_2st_MA],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,2,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,2,5,None,None],
        'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,None,None,None],
        'fit_2st_rainbow_midpoint:[fittype,pars_jk_meff2st]':[fittype,pars_jk_meff2st],
        'xunit':yu.ens2a[ens], 'yunit':yunit,
    }
    
    dic_sum={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,fits_sum,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,2,2,None,None],
        'xunit':yu.ens2a[ens], 'yunit':yunit,
        'shift:[rainbow,midpoint,fit]':[0,0,-0.2],
    }
    return dic,dic_sum

js_plt=['j+;conn','j-;conn']; enss_plt=enss
# enss_plt=['e']

for ij,j in enumerate(['j-;conn']):
    print(f'{ij}/{len(js_plt)}',j,end='                 \r')
    keys=[(ens,j) for ens in enss_plt]

    t=[createDic(key) for key in keys]
    list_dic=[ele[0] for ele in t]
    list_dic_sum=[ele[1] for ele in t]

    fig,axs=yu.makePlot_3pt(list_dic,shows=['rainbow','midpoint','fit_2st'],noLegendQ=True,colHeaders=None,sharey=True,fullband='fit_2st',oddmidQ=True)
    fig,axs=yu.makePlot_3pt(list_dic_sum,shows=['rainbow','rainbow','fit_sum'],figAxs=(fig,axs),colors_fit=['g'],fmts_fit=['o'],colHeaders=None)
    
    for i in range(len(enss)):
        axs[i,0].set_ylabel(r'$\tilde{A}_{20}^{u-d}(0)$')
        
    axs[0,0].set_ylim([0.08,0.32])
    
    # axs[-1,2].set_xlim([0.35,1.45])
    # axs[-1,2].set_xticks(np.arange(0.4,1.5,0.2))
    # yu.addRowHeader(axs,[yu.ens2label[ens][0] for ens in enss_plt])
    yu.setpath('plot_paper')
    yu.finalizePlot(f'rainbow_avgx_2/{j}_tilde' if j in ['j-;conn'] else f'rainbow/{j}_bare_mu=nu',mkdirQ=True)
    yu.setpath('analysis_avgx_2')

# disc

In [15]:
ens2c2pt={}; ens2moms_2pt={}; ens2c2pt0={}; ens2Njk={}
for ens in enss:
    if ens in ['e']:
        basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
        path=f'{basepath}disc_2pt_N=100,127,84_sup.h5'
        with h5py.File(path) as f:
            moms_2pt=yu.moms2list(f['moms'])
            c2pt=yu.jackknife(np.real(f['data/N_N'][:,:,:]))
            c2pt=yu.superjackknife(c2pt,yum.key2cfgs[(ens,'N')],yum.key2cfgs[(ens,'all')])
    else:
        basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
        path=f'{basepath}disc_2pt.h5'
        with h5py.File(path) as f:
            moms_2pt=yu.moms2list(f['moms'])
            c2pt=yu.jackknife(np.real(f['data/N_N'][:,:,:]))
        
    ens2moms_2pt[ens]=moms_2pt
    ens2c2pt[ens]=c2pt
    ens2c2pt0[ens]=c2pt[:,:,moms_2pt.index([0,0,0])]
    ens2Njk[ens]=len(c2pt)

key2tf2ratio={}
stouts=[5,7,10,13,15,20,25,30,35,40]
# stouts=range(0,41)
js=['j+;disc','js;disc','jc;disc']+[f'jg;stout{stout}' for stout in stouts]
for ens in enss:
    if ens in ['e']:
        path=f'/p/project1/ngff/li47/code/scratch/run/05_moments_run5_1DV/{yu.ens2full[ens]}/data_merge_sup/disc_0,0,1,0,0,0.h5'
        with h5py.File(path) as f:
            for jtf in f['data'].keys():
                j,tf=jtf.split('_'); tf=int(tf)
                if j not in js:
                    continue        
                key=(ens,j)
                if key not in key2tf2ratio:
                    key2tf2ratio[key]={}
                c3pt=yu.jackknife(f['data'][jtf][:,:,0,projs.index('P0'),inserts.index('tz')])
                c3pt=yu.superjackknife(c3pt,yum.key2cfgs[(ens,yum.j2jkey(j))],yum.key2cfgs[(ens,'all')])
                c2pt=ens2c2pt[ens][:,tf,ens2moms_2pt[ens].index([0,0,1])]
                factor=1/(1j*2*np.pi/yu.ens2NL[ens])
                ratio=np.real(c3pt/c2pt[:,None]*factor)
                key2tf2ratio[key][tf]=ratio
    else:
        path=f'/p/project1/ngff/li47/code/scratch/run/05_moments_run5_1DV/{yu.ens2full[ens]}/data_merge/disc_0,0,1,0,0,0.h5'
        with h5py.File(path) as f:
            for jtf in f['data'].keys():
                j,tf=jtf.split('_'); tf=int(tf)
                if j not in js:
                    continue        
                key=(ens,j)
                if key not in key2tf2ratio:
                    key2tf2ratio[key]={}
                c3pt=yu.jackknife(f['data'][jtf][:,:,0,projs.index('P0'),inserts.index('tz')])
                c2pt=ens2c2pt[ens][:,tf,ens2moms_2pt[ens].index([0,0,1])]
                factor=1/(1j*2*np.pi/yu.ens2NL[ens])
                ratio=np.real(c3pt/c2pt[:,None]*factor)
                key2tf2ratio[key][tf]=ratio
            
ens2tfs_disc={}
for ens in enss:
    tfs=list(key2tf2ratio[(ens,'j+;disc')].keys()); tfs.sort()
    ens2tfs_disc[ens]=tfs
    print(ens,tfs[0],tfs[-1])
    
path='pkl/analysis_c2pt/reg_ignore/ens2pars_jk_meffnst_selected.pkl'
[ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]=yu.load_pkl(path)

b 2 22
c 2 26
d 2 30
e 2 32


In [16]:
selections=['bandfit_WA','const_MA','sum_MA','2st_MA']
selection2key2bare={selection:{} for selection in selections}

overwrite=False
def createDic(key):
    rainbow_tfphy_min=0.5
    rainbow_tfphy_max=1.1
    sum_tfphy_max=0.7
    dt=2
    
    ens,j=key
    
    gett=lambda t:round(t/yu.ens2a[ens])
    def get_tfs(tmin,tmax,dt=1):
        return range(gett(tmin),gett(tmax),dt)
    lat_a=yu.ens2a[ens]
    
    tfmins_1st=get_tfs(0.3,1.25)
    tcmins_1st=get_tfs(2*lat_a,0.6)
    tfmins_sum=get_tfs(0.2,1.25); tcmins_sum=[2]
    
    tf2ratio=yu.cut_tf2ratio(key2tf2ratio[(ens,j)],gett(1.3))
    
    fits_const=yu.doFits_3pt('const',tf2ratio,tfmins_1st,tcmins_1st,symmetrizeQ=True,corrQ=True,label=f'{ens}_{j}_const',overwrite=overwrite)
    (tfphy,tcphy)=tftcphy_A20_discq
    fits=fits_const
    fitlabels=[fit[0] for fit in fits]
    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    tcmin=fitlabels[ind][1]
    # print(tcmin)    
    fits_const=[fit for fit in fits_const if fit[0][1]==tcmin]
    fit_const_MA=yu.doMA_3pt(fits_const,fitlabels=fitlabels[ind])
    
    fits_band=yu.doFits_3pt_band(tf2ratio,tcmins_1st,corrQ=False,label=f'{ens}_{j}_band')
    # print(tcmin,[fit[0][1] for fit in fits_band])
    fits_band=[fit for fit in fits_band if fit[0][1]==tcmin]
    # fit_band_WA=yu.doWA_band(fits_band,tf_min=gett(0.7),tf_max=gett(1.3),tcmin=gett(0.2),corrQ=False)
    
    fits_sum=yu.doFits_3pt('sum',tf2ratio,tfmins_sum,tcmins_sum,corrQ=False,label=f'{ens}_{j}_sum',overwrite=overwrite)
    fits_sum=[fit for fit in fits_sum if fit[0][1]==2]
    # fits=fits_sum
    # fitlabels=[fit[0] for fit in fits]
    # ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    # fit_sum_MA=yu.doMA_3pt(fits_sum,fitlabels=fitlabels[ind])
    
    # selection2key2bare['bandfit_WA'][key]=fit_band_WA[0][:,0]
    selection2key2bare['const_MA'][key]=fit_const_MA[0][:,0]
    # selection2key2bare['sum_MA'][key]=fit_sum_MA[0][:,0]
    
    dic={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,fits_band,fits_const,fits_sum,None],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,fit_const_MA,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[gett(rainbow_tfphy_min),gett(rainbow_tfphy_max),2,dt],
        'fit_band:[tfmin,tfmax,tcmin_min,tcmin_max,dtf,dtc]':[gett(rainbow_tfphy_min),gett(rainbow_tfphy_max),None,None,dt,None],
        'fit_const:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[gett(rainbow_tfphy_min),None,None,None,None,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,None,None,None],
        'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,None,None,None],
        'xunit':yu.ens2a[ens],
    }
    dic_sum={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,fits_sum,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[gett(rainbow_tfphy_min),gett(sum_tfphy_max),2,2,None,None],
        'xunit':yu.ens2a[ens],
        'shift:[rainbow,midpoint,fit]':[0,0,-0.2],
    }
    return dic,dic_sum

def createDic2(key):
    rainbow_tfphy_min=0.5
    rainbow_tfphy_max=1.1
    sum_tfphy_max=0.7
    dt=2
    
    ens,j=key
    gett=lambda t:round(t/yu.ens2a[ens])
    def get_tfs(tmin,tmax,dt=1):
        return range(gett(tmin),gett(tmax),dt)
    lat_a=yu.ens2a[ens]
    
    tfmins_1st=get_tfs(0.3,1.25)
    tcmins_1st=get_tfs(2*lat_a,0.6)
    tfmins_sum=get_tfs(0.2,1.25); tcmins_sum=[2]
    
    tf2ratio=yu.cut_tf2ratio(key2tf2ratio[(ens,j)],gett(1.3))
    
    fits_const=yu.doFits_3pt('const',tf2ratio,tfmins_1st,tcmins_1st,symmetrizeQ=True,corrQ=True,label=f'{ens}_{j}_const',overwrite=overwrite)
    (tfphy,tcphy)=tftcphy_A20_gluon
    fits=fits_const
    fitlabels=[fit[0] for fit in fits]
    ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    tcmin=fitlabels[ind][1]
    fits_const=[fit for fit in fits_const if fit[0][1]==tcmin]
    fit_const_MA=yu.doMA_3pt(fits_const,fitlabels=fitlabels[ind])
    
    fits_band=yu.doFits_3pt_band(tf2ratio,tcmins_1st,corrQ=False,label=f'{ens}_{j}_band')
    fits_band=[fit for fit in fits_band if fit[0][1]==tcmin]
    # fit_band_WA=yu.doWA_band(fits_band,tf_min=gett(0.7),tf_max=gett(1.3),tcmin=gett(0.2),corrQ=False)
    
    fits_sum=yu.doFits_3pt('sum',tf2ratio,tfmins_sum,tcmins_sum,corrQ=False,label=f'{ens}_{j}_sum',overwrite=overwrite)
    fits_sum=[fit for fit in fits_sum if fit[0][1]==2]
    # fits=fits_sum
    # fitlabels=[fit[0] for fit in fits]
    # ind=np.argmin([np.abs(tfmin*yu.ens2a[ens] - tfphy) + np.abs(tcmin*yu.ens2a[ens] - tcphy) for tfmin,tcmin in fitlabels])
    # fit_sum_MA=yu.doMA_3pt(fits_sum,fitlabels=fitlabels[ind])
    
    # selection2key2bare['bandfit_WA'][key]=fit_band_WA[0][:,0]
    selection2key2bare['const_MA'][key]=fit_const_MA[0][:,0]
    # selection2key2bare['sum_MA'][key]=fit_sum_MA[0][:,0]
    
    dic={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,fits_band,fits_const,fits_sum,None],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,fit_const_MA,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[gett(rainbow_tfphy_min),gett(rainbow_tfphy_max),2,dt],
        'fit_band:[tfmin,tfmax,tcmin_min,tcmin_max,dtf,dtc]':[gett(rainbow_tfphy_min),gett(rainbow_tfphy_max),None,None,dt,None],
        'fit_const:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[gett(rainbow_tfphy_min),None,None,None,None,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,None,None,None],
        'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,None,None,None],
        'xunit':yu.ens2a[ens],
    }
    dic_sum={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,fits_sum,None],
        'fit_sum:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[gett(rainbow_tfphy_min),gett(sum_tfphy_max),2,2,None,None],
        'xunit':yu.ens2a[ens],
        'shift:[rainbow,midpoint,fit]':[0,0,-0.2],
    }
    return dic,dic_sum

# js_plt=['j+;disc']; enss_plt=['b','c','d']
# js_plt=['jg;stout10']; enss_plt=['b','c','d']
js_plt=js; enss_plt=enss

for ij,j in enumerate(['j+;disc','js;disc','jc;disc','jg;stout10']):
    # print(f'{ij}/{len(js_plt)}',j,end='                 \r')
    keys=[(ens,j) for ens in enss_plt]

    t=[createDic2(key) if key[1].startswith('jg') else createDic(key) for key in keys]
    list_dic=[ele[0] for ele in t]
    list_dic_sum=[ele[1] for ele in t]

    fig,axs=yu.makePlot_3pt(list_dic,shows=['rainbow', 'fit_band', 'fit_const'],colHeaders=None,noLegendQ=True,fullband='fit_const')
    jstr=j[1]
    jstr='u+d' if jstr=='+' else jstr
    for i in range(len(enss)):
        axs[i,0].set_ylabel(rf'$\tilde{{A}}_{{20}}^{{{jstr}}}(0)$')
        axs[i,0].set_ylim(axs[i,0].get_ylim())
    fig,axs=yu.makePlot_3pt(list_dic_sum,shows=['rainbow','fit_band','fit_sum'],figAxs=(fig,axs),colors_fit=['g'],fmts_fit=['o'],colHeaders=None)
    # yu.addRowHeader(axs,[yu.ens2label[ens][0] for ens in enss_plt])
    axs[-1,0].set_xticks([-0.3,0,0.3])
    axs[-1,1].set_xticks([0.7,1.0])
    axs[-1,2].set_xticks([0.7,1.0])
    # axs[-1,2].set_xlim([0.35,1.45])
    # axs[-1,2].set_xticks(np.arange(0.4,1.5,0.2))
        
    yu.setpath('plot_paper')
    yu.finalizePlot(f'rainbow_avgx_2/{j}_tilde',mkdirQ=True)
    yu.setpath('analysis_avgx_2')